In [1]:
import json
import pandas as pd

import json

file_path = "/home/shreya/sim/data/raw/openfda/drug_label/63355267-a151-4d7d-9d6e-0a9ac9b5a48f/pages.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    first_line = json.loads(f.readline())

results = first_line["results"]

print(results[0])

{'effective_time': '20210902', 'inactive_ingredient': ['INACTIVE INGREDIENTS Sucrose'], 'purpose': ['USES USES: Temporary Relief - Acne, Boils* * Claims based on traditional homeopathic practice, not accepted medical evidence. Not FDA evaluated.'], 'keep_out_of_reach_of_children': ['Keep this and all medication out of reach of children'], 'warnings': ['WARNINGS This product is to be used for self-limiting conditions If symptoms do not improve in 4 days, or worsen, discontinue use and seek assistance of health professional. As with any drug, if you are pregnant, or nursing a baby, seek professional advice before taking this product. Keep this and all medication out of reach of children Do not use if capseal is broken or missing. Close the cap tightly after use.'], 'questions': ['QUESTIONS OR COMMENTS www.Rxhomeo.com | 1.888.2796642 | info@rxhomeo.com Rxhomeo, Inc 3200 Commander Dr, Ste 100-W1, Carrollton, TX 75006 USA'], 'spl_product_data_elements': ['SILICEA SILICEA SUCROSE SILICON DIO

In [2]:
len(results)

1000

In [3]:
df_one = pd.json_normalize(results)

print("Rows:", df_one.shape[0])
print("Columns:", df_one.shape[1])

Rows: 1000
Columns: 130


In [1]:
import json
import pandas as pd

import json

file_path = "/home/shreya/sim/data/raw/openfda/drug_label/33c61f02-49ad-4f93-a4ec-a3ed5cb0cb7d/pages.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    first_line = json.loads(f.readline())

results = first_line["results"]

print(results[0])

{'spl_product_data_elements': ['Emvita 23 Agaricus muscarius, Cuprum met, Hypophysis, Rhus tox ALCOHOL WATER AMANITA MUSCARIA FRUITING BODY AMANITA MUSCARIA FRUITING BODY COPPER COPPER SUS SCROFA PITUITARY GLAND SUS SCROFA PITUITARY GLAND TOXICODENDRON PUBESCENS LEAF TOXICODENDRON PUBESCENS LEAF'], 'active_ingredient': ['Drug Facts Active Ingredients: (HPUS*) 25% of each Agaricus muscarius 21X Cuprum met 800C Hypophysis 21X Rhus tox 18LM *The letters "HPUS" indicate that the components in this product are officially monographed in the Homeopathic Pharmacopoeia of the United States. †Claims based on traditional homeopathic practice, not accepted medical evidence. Not FDA evaluated.'], 'indications_and_usage': ['Uses: (†) Homeopathic remedy for general well being: tenseness.'], 'warnings': ['Warnings: Stop use if symptoms persist or worsen. If you are pregnant or breastfeeding, consult a healthcare professional prior to use. Keep out of reach of children.'], 'keep_out_of_reach_of_childre

In [2]:
import os
import joblib
import pandas as pd

file_path = "./openfda.pkl"

# your new dataframe
df_one = pd.json_normalize(results)

print("Rows:", df_one.shape[0])
print("Columns:", df_one.shape[1])

# ensure directory exists
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# append logic
if os.path.exists(file_path):
    existing_df = joblib.load(file_path)
    combined_df = pd.concat([existing_df, df_one], ignore_index=True, sort=False)
    print("Appending to existing file...")
else:
    print("Creating new file...")
    combined_df = df_one

# save back
joblib.dump(combined_df, file_path)

print("Final shape:", combined_df.shape)
print("Saved successfully!")

Rows: 1000
Columns: 134
Appending to existing file...
Final shape: (2000, 141)
Saved successfully!


In [4]:
print("Unique spl_id:", combined_df['openfda.spl_id'].astype(str).nunique())


Unique spl_id: 750


In [5]:
df_one = combined_df.copy()

In [6]:
sparsity = df_one.isnull().mean() * 100
sparse_cols = sparsity[sparsity >= 99].index
print("Number of highly sparse columns:", len(sparse_cols))

Number of highly sparse columns: 39


In [7]:
sparsity = df_one.isnull().mean() * 100
sparse_cols = sparsity[sparsity >= 98].index
print("Number of highly sparse columns:", len(sparse_cols))

Number of highly sparse columns: 43


In [44]:
import joblib
df_one = joblib.load("./openfda.pkl")   
df_one.shape

(2000, 141)

In [3]:
df_one.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Columns: 141 entries, effective_time to do_not_use_table
dtypes: object(141)
memory usage: 2.2+ MB


In [46]:
sparsity = df_one.isnull().mean() * 100

# columns BETWEEN 98% and 99%
mid_sparse_cols = sparsity[(sparsity >= 98) & (sparsity < 99)].index

print("Number of columns between 98% and 99% sparsity:", len(mid_sparse_cols))

# non-null counts for these columns
non_null_counts = df_one[mid_sparse_cols].notnull().sum()

print("\nNon-null values in these columns:")
print(non_null_counts)

# total non-null values across these columns
print("\nTotal non-null values:", non_null_counts.sum())

Number of columns between 98% and 99% sparsity: 4

Non-null values in these columns:
active_ingredient_table    39
purpose_table              32
precautions_table          22
pharmacodynamics_table     24
dtype: int64

Total non-null values: 117


In [47]:
sparsity = df_one.isnull().mean() * 100

# columns with >= 99% null
cols_to_drop = sparsity[sparsity >= 99].index

print("Columns to drop:", len(cols_to_drop))

# drop them
df_one = df_one.drop(columns=cols_to_drop)

print("New shape:", df_one.shape)

Columns to drop: 39
New shape: (2000, 102)


In [48]:
sparsity = df_one.isnull().mean() * 100

# columns with >= 99% null
cols_to_drop = sparsity[sparsity >= 95].index

print("Columns to drop:", len(cols_to_drop))


Columns to drop: 20


In [49]:
df_one.columns.to_list()

['effective_time',
 'inactive_ingredient',
 'purpose',
 'keep_out_of_reach_of_children',
 'warnings',
 'questions',
 'spl_product_data_elements',
 'version',
 'dosage_and_administration',
 'pregnancy_or_breast_feeding',
 'stop_use',
 'storage_and_handling',
 'do_not_use',
 'package_label_principal_display_panel',
 'indications_and_usage',
 'set_id',
 'id',
 'active_ingredient',
 'openfda.brand_name',
 'openfda.generic_name',
 'openfda.manufacturer_name',
 'openfda.product_ndc',
 'openfda.product_type',
 'openfda.route',
 'openfda.substance_name',
 'openfda.spl_id',
 'openfda.spl_set_id',
 'openfda.package_ndc',
 'openfda.is_original_packager',
 'openfda.upc',
 'openfda.unii',
 'when_using',
 'ask_doctor',
 'openfda.application_number',
 'openfda.rxcui',
 'openfda.nui',
 'openfda.pharm_class_epc',
 'openfda.pharm_class_cs',
 'spl_unclassified_section',
 'description',
 'clinical_pharmacology',
 'clinical_pharmacology_table',
 'pharmacokinetics',
 'microbiology',
 'clinical_studies',
 's

In [50]:
df_one['clinical_pharmacology'].equals(df_one['clinical_pharmacology_table'])

False

In [51]:
comparison = df_one[['clinical_pharmacology', 'clinical_pharmacology_table']]

# rows where values differ
diff_rows = comparison[
    (comparison['clinical_pharmacology'] != comparison['clinical_pharmacology_table']) &
    ~(comparison['clinical_pharmacology'].isnull() & comparison['clinical_pharmacology_table'].isnull())
]

print("Number of different rows:", len(diff_rows))
print(diff_rows)

Number of different rows: 759
                                  clinical_pharmacology  \
4     [CLINICAL PHARMACOLOGY Pharmacokinetics: Serum...   
5     [12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Ac...   
12    [12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Ac...   
13    [CLINICAL PHARMACOLOGY Mechanism of Action The...   
14    [12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Ac...   
...                                                 ...   
1980  [12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Ac...   
1986  [12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Ac...   
1987  [CLINICAL PHARMACOLOGY Calcium is the fifth mo...   
1988  [12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Ac...   
1992  [12 CLINICAL PHARMACOLOGY 12.1 Mechanism of Ac...   

                            clinical_pharmacology_table  
4     [<table width="100%" styleCode="Noautorules"><...  
5                                                   NaN  
12                                                  NaN  
13    [<table> <caption> TABL

In [52]:
print("Text present:", df_one['clinical_pharmacology'].notnull().sum())
print("Table present:", df_one['clinical_pharmacology_table'].notnull().sum())

Text present: 759
Table present: 241


In [53]:
cols = df_one.columns

# find _table columns where base column exists
table_cols_to_drop = [
    col for col in cols 
    if col.endswith('_table') and col.replace('_table', '') in cols
]

print("Dropping these _table columns:", len(table_cols_to_drop))

# drop them
df_one = df_one.drop(columns=table_cols_to_drop)

print("New shape:", df_one.shape)

Dropping these _table columns: 18
New shape: (2000, 84)


In [54]:
df_one.columns.to_list()

['effective_time',
 'inactive_ingredient',
 'purpose',
 'keep_out_of_reach_of_children',
 'warnings',
 'questions',
 'spl_product_data_elements',
 'version',
 'dosage_and_administration',
 'pregnancy_or_breast_feeding',
 'stop_use',
 'storage_and_handling',
 'do_not_use',
 'package_label_principal_display_panel',
 'indications_and_usage',
 'set_id',
 'id',
 'active_ingredient',
 'openfda.brand_name',
 'openfda.generic_name',
 'openfda.manufacturer_name',
 'openfda.product_ndc',
 'openfda.product_type',
 'openfda.route',
 'openfda.substance_name',
 'openfda.spl_id',
 'openfda.spl_set_id',
 'openfda.package_ndc',
 'openfda.is_original_packager',
 'openfda.upc',
 'openfda.unii',
 'when_using',
 'ask_doctor',
 'openfda.application_number',
 'openfda.rxcui',
 'openfda.nui',
 'openfda.pharm_class_epc',
 'openfda.pharm_class_cs',
 'spl_unclassified_section',
 'description',
 'clinical_pharmacology',
 'pharmacokinetics',
 'microbiology',
 'clinical_studies',
 'contraindications',
 'precautions

In [55]:
def clean_drug_name(x):
    if isinstance(x, list):
        return ', '.join(x)   # convert list → string
    return x

df_one['drug_name_clean'] = df_one['openfda.generic_name'].apply(clean_drug_name)

# now apply string operations
df_one['drug_name_clean'] = (
    df_one['drug_name_clean']
    .astype(str)
    .str.lower()
    .str.strip()
)

In [56]:
df_one.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 85 columns):
 #   Column                                                      Non-Null Count  Dtype 
---  ------                                                      --------------  ----- 
 0   effective_time                                              2000 non-null   object
 1   inactive_ingredient                                         1212 non-null   object
 2   purpose                                                     1185 non-null   object
 3   keep_out_of_reach_of_children                               1183 non-null   object
 4   warnings                                                    1576 non-null   object
 5   questions                                                   652 non-null    object
 6   spl_product_data_elements                                   1999 non-null   object
 7   version                                                     2000 non-null   object
 8   dosage_a

In [57]:
def clean_val(x):
    if isinstance(x, list):
        return ', '.join(x)
    return x

# clean all relevant columns first
cols = [
    'openfda.generic_name',
    'openfda.brand_name',
    'openfda.substance_name',
    'active_ingredient'
]

for col in cols:
    df_one[col] = df_one[col].apply(clean_val)

# now create strong drug_name column
df_one['drug_name_clean'] = (
    df_one['openfda.generic_name']
    .fillna(df_one['openfda.brand_name'])
    .fillna(df_one['openfda.substance_name'])
    .fillna(df_one['active_ingredient'])
)

# normalize
df_one['drug_name_clean'] = (
    df_one['drug_name_clean']
    .astype(str)
    .str.lower()
    .str.strip()
)

# remove fake "nan" strings
df_one['drug_name_clean'] = df_one['drug_name_clean'].replace('nan', None)

In [58]:
df_one['drug_name_clean']

0                                                 silicea
1       bronze active ingredients: titanium dioxide 2 ...
2                                         povidone-iodine
3                             active ingredients mezereum
4                                                    None
                              ...                        
1995                     camphor, eucalyptus oil, menthol
1996    active ingredient (in each lozenge) hexylresco...
1997    active ingredients: fucus vesiculosus 2x, caps...
1998                                             glycerin
1999                                 brimonidine tartrate
Name: drug_name_clean, Length: 2000, dtype: object

In [59]:
import re
import pandas as pd
def clean_text(x):
    if pd.isna(x):
        return None
    
    x = str(x).lower()
    
    # remove common prefixes
    x = re.sub(r'active ingredients?:?', '', x)
    x = re.sub(r'active ingredient \(.*?\)', '', x)
    
    return x.strip()

df_one['drug_name_clean'] = df_one['drug_name_clean'].apply(clean_text)

In [60]:
import re

def advanced_clean(x):
    if pd.isna(x):
        return None
    
    x = str(x).lower()
    
    # remove brackets content
    x = re.sub(r'\(.*?\)', '', x)
    
    # remove words like bronze, base, etc.
    x = re.sub(r'\b(bronze|base|formula|ingredients?)\b', '', x)
    
    return x.strip()

df_one['drug_name_clean'] = df_one['drug_name_clean'].apply(advanced_clean)

In [61]:
def remove_noise(x):
    if pd.isna(x):
        return None
    
    # remove numbers + units + potency like 2x, 3x
    x = re.sub(r'\d+(\.\d+)?\s*(mg|ml|g|%|mcg|x)?', '', x)
    
    return x.strip()

df_one['drug_name_clean'] = df_one['drug_name_clean'].apply(remove_noise)

In [62]:
df_one['drug_name_clean'] = df_one['drug_name_clean'].str.split(',')

df_one = df_one.explode('drug_name_clean')

df_one['drug_name_clean'] = df_one['drug_name_clean'].str.strip()

In [63]:
df_one['drug_name_clean'] = df_one['drug_name_clean'].replace('', None)

df_one = df_one.dropna(subset=['drug_name_clean'])

# remove very short junk
df_one = df_one[df_one['drug_name_clean'].str.len() > 2]

In [64]:
junk = ['water', 'flavor', 'fragrance', 'color']

df_one = df_one[~df_one['drug_name_clean'].isin(junk)]

In [65]:
df_one['drug_name_clean']

0                           silicea
1                  titanium dioxide
1       ethylhexyl methoxycinnamate
1                        zinc oxide
2                   povidone-iodine
                   ...             
1997             calcarea carbonica
1997                      graphites
1997                kaki carbonicum
1998                       glycerin
1999           brimonidine tartrate
Name: drug_name_clean, Length: 2709, dtype: object

In [66]:
df_one.shape

(2709, 85)

In [67]:
import joblib

joblib.dump(df_one, "openfda_v1.pkl")

['openfda_v1.pkl']

In [68]:
df_one.to_csv("openfda_v1.csv", index=False)

In [69]:
df_one.head()

,effective_time,inactive_ingredient,purpose,keep_out_of_reach_of_children,warnings,questions,spl_product_data_elements,version,dosage_and_administration,pregnancy_or_breast_feeding,...,teratogenic_effects,animal_pharmacology_and_or_toxicology,openfda.pharm_class_pe,labor_and_delivery,controlled_substance,dependence,ask_doctor_or_pharmacist,other_safety_information,abuse,drug_name_clean
0,20210902,[INACTIVE INGREDIENTS Sucrose],"[USES USES: Temporary Relief - Acne, Boils* * ...",[Keep this and all medication out of reach of ...,[WARNINGS This product is to be used for self-...,[QUESTIONS OR COMMENTS www.Rxhomeo.com | 1.888...,[SILICEA SILICEA SUCROSE SILICON DIOXIDE SILIC...,2,"[DOSAGE Adults- Take 4 or 6 Pellets by mouth, ...","[As with any drug, if you are pregnant, or nur...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,silicea
1,20150109,"[INGREDIENTS: TALC, POLYMETHYL METHACRYLATE, V...",[Purpose Sunscreen],[Keep out of reach of children If product is s...,[Warnings For external use only.],NaN,[CHANTECAILLE PROTECTION NATURELLE BRONZE SPF ...,4,[Directions Protection Naturelle SPF 46 PA+++ ...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,titanium dioxide
1,20150109,"[INGREDIENTS: TALC, POLYMETHYL METHACRYLATE, V...",[Purpose Sunscreen],[Keep out of reach of children If product is s...,[Warnings For external use only.],NaN,[CHANTECAILLE PROTECTION NATURELLE BRONZE SPF ...,4,[Directions Protection Naturelle SPF 46 PA+++ ...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ethylhexyl methoxycinnamate
1,20150109,"[INGREDIENTS: TALC, POLYMETHYL METHACRYLATE, V...",[Purpose Sunscreen],[Keep out of reach of children If product is s...,[Warnings For external use only.],NaN,[CHANTECAILLE PROTECTION NATURELLE BRONZE SPF ...,4,[Directions Protection Naturelle SPF 46 PA+++ ...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,zinc oxide
2,20250102,"[Inactive ingredients pareth 25-9, purified wa...",[Purpose First aid Antiseptic],"[Keep out of reach of children If swallowed, g...",[Warnings For external use only],NaN,[Betadine POVIDONE-IODINE POVIDONE-IODINE IODI...,1,[Directions clean the affected area spray a sm...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,povidone-iodine


In [71]:
from pprint import pprint

pprint(df_one.iloc[0].to_dict())

{'abuse': nan,
 'active_ingredient': 'ACTIVE INGREDIENT SILICEA HPUS 2X and higher',
 'adverse_reactions': nan,
 'animal_pharmacology_and_or_toxicology': nan,
 'ask_doctor': nan,
 'ask_doctor_or_pharmacist': nan,
 'boxed_warning': nan,
 'carcinogenesis_and_mutagenesis_and_impairment_of_fertility': nan,
 'clinical_pharmacology': nan,
 'clinical_studies': nan,
 'contraindications': nan,
 'controlled_substance': nan,
 'dependence': nan,
 'description': nan,
 'do_not_use': ['Do not use if capseal is broken or missing. Close the cap '
                'tightly after use.'],
 'dosage_and_administration': ['DOSAGE Adults- Take 4 or 6 Pellets by mouth, '
                               'three times daily or as suggested by '
                               'physician. Children 2 years and older- take '
                               '1/2 the adult dose.'],
 'dosage_forms_and_strengths': nan,
 'drug_abuse_and_dependence': nan,
 'drug_and_or_laboratory_test_interactions': nan,
 'drug_interactions':

In [2]:
import pandas as pd
df_one=pd.read_csv("./openfda_v1.csv")
df_one.shape

(2709, 85)

In [5]:
df_one['drug_name_clean'].nunique()

1475